# Docker Bootcamp for Python Developers

This notebook contains **30 Docker lessons**. Each lesson has a Markdown explanation followed by a Python cell containing commented examples, Docker commands, or configuration snippets.

## Topics
- Images, containers, tags, registries, and lifecycle
- Docker CLI, logs, exec, ports, volumes, bind mounts, environment variables
- Dockerfiles, build context, `.dockerignore`, build caching, and image tags
- Docker Compose, networks, service dependencies, health checks, and persistence
- Security basics, non-root users, multi-stage builds, resources, registries, debugging, cleanup, and CI/CD
- A companion **Flask + Redis** practice application

## Prerequisites
Install Docker Desktop on Windows/macOS or Docker Engine plus the Compose plugin on Linux. Run `docker version` in your terminal to check that both the CLI and daemon are available.

**The notebook does not automatically run Docker CLI commands.** Command examples are displayed for review and manual execution. This avoids accidentally starting, stopping, or deleting resources while reading the notebook.

## Run the companion practice app
Extract the ZIP pack. Open a terminal in `docker_practice_project/` and run:

```bash
docker compose up --build
```

Then open `http://localhost:5000/`, `http://localhost:5000/health`, and `http://localhost:5000/visits`.

`docker compose down -v` deletes the Compose-managed named volume and its data. Do not use it unless you intentionally want to erase the practice data.


## Lesson 1: 1. Docker: the big picture

Docker builds images and runs them as containers. An **image** is a layered template; a **container** is an instance of an image. A registry stores images. Volumes store data beyond a container's writable layer, networks connect services, and Compose describes multi-container applications.

Containers share the host operating system's kernel, unlike traditional virtual machines that run a separate guest OS kernel. Containers still need careful security, data management, updates, and monitoring; Docker does not automatically provide those.

In [ ]:
concepts = {  # Define the basic Docker vocabulary.
    "Image": "A layered template used to create containers.",  # The packaged app and runtime files.
    "Container": "An isolated process and its runtime configuration.",  # A running or stopped instance.
    "Registry": "A place to store and distribute images.",  # For example, a private image registry.
    "Volume": "Docker-managed storage that can outlive a container.",  # A place to keep persistent state.
    "Network": "Virtual connectivity between containers and external endpoints.",  # Enables service-to-service communication.
    "Compose": "A YAML-based way to define and run multiple services.",  # Coordinates a local application stack.
}  # Finish the glossary.
for name, meaning in concepts.items():  # Iterate through Docker concepts.
    print(f"{name}: {meaning}")  # Display each concept and its meaning.

## Lesson 2: 2. Check the Docker CLI and daemon

`docker --version` checks the command-line client. `docker version` reports client and server details; `docker info` reports daemon configuration. The CLI may be installed even when the Docker daemon is not running, especially if Docker Desktop has not started.

The example below checks the CLI without creating or deleting resources. If the Docker command is unavailable, install Docker Desktop or Docker Engine and ensure its daemon is running.

In [ ]:
import shutil  # Locate commands on the current PATH.
import subprocess  # Run a read-only diagnostic.

docker_executable = shutil.which("docker")  # Search for the Docker CLI.
print("Docker CLI:", docker_executable or "not found")  # Report whether it is installed.
if docker_executable:  # Only run the diagnostic if Docker is available.
    result = subprocess.run(["docker", "--version"], capture_output=True, text=True, timeout=5)  # Ask the CLI for its version.
    print(result.stdout.strip() or result.stderr.strip())  # Display the result or diagnostic.
    print("Exit code:", result.returncode)  # Zero normally means the command succeeded.

## Lesson 3: 3. Images, tags, and registries

Images are built from layers and referred to by names and tags such as `python:3.12-slim`. A tag is convenient but may move to a newer build, so it is not always a fully immutable reference. Registries store and distribute images.

Use clear version tags for project releases. In controlled production pipelines, consider pinning base images by digest and updating them deliberately. Pulling an image downloads it and requires registry access.

In [ ]:
image_commands = [  # Common image commands to practice in a terminal.
    "docker image ls",  # List local images.
    "docker pull python:3.12-slim",  # Download the Python base image.
    "docker image inspect python:3.12-slim",  # Inspect image configuration and metadata.
    "docker image history python:3.12-slim",  # Inspect image layer history.
]  # Finish the command list.
print("\n".join(image_commands))  # Display the commands without executing them automatically.
print("Tip: use a deliberate version tag instead of relying only on the mutable 'latest' tag.")  # Summarize tagging advice.

## Lesson 4: 4. Container lifecycle

`docker run` creates and starts a container. Other common commands list, stop, start, inspect, and remove containers. A container's writable layer is not a reliable place for data that must survive replacement; use a volume or an external data service for persistent state.

`--rm` removes a disposable container when its process exits. It does not remove a named volume attached to the container.

In [ ]:
lifecycle_commands = [  # A basic container lifecycle, for manual practice.
    "docker run --rm hello-world",  # Run and automatically remove a small test container.
    "docker ps",  # List running containers.
    "docker ps -a",  # Include stopped containers.
    "docker stop <container-name-or-id>",  # Ask a running container to stop.
    "docker start <stopped-container-name-or-id>",  # Start an existing stopped container.
    "docker rm <stopped-container-name-or-id>",  # Remove a stopped container.
]  # Finish the lifecycle examples.
print("\n".join(lifecycle_commands))  # Print the commands for review.
print("Inspect the actual container name and state before removing it.")  # Add a safety reminder.

## Lesson 5: 5. Names, interactive mode, and default commands

`--name` gives a container a human-readable name. `-it` combines interactive input with a pseudo-terminal. Text after the image name can override the image's default command.

The image and the running container are distinct: you can create multiple containers from one image, each with different names, ports, environment variables, and storage mounts.

In [ ]:
run_examples = [  # Show useful run modes.
    "docker run --rm -it python:3.12-slim python",  # Open an interactive Python prompt.
    "docker run --rm --name temp-shell -it python:3.12-slim sh",  # Open a shell in a disposable named container.
    "docker run --name demo-nginx -d -p 127.0.0.1:8080:80 nginx:alpine",  # Start a small web server in the background.
    "docker logs demo-nginx",  # Read its output.
    "docker stop demo-nginx",  # Stop it when finished.
    "docker rm demo-nginx",  # Remove the stopped demo container.
]  # Finish examples.
print("\n".join(run_examples))  # Display commands that can be run individually.

## Lesson 6: 6. Logs, exec, inspect, and stats

`docker logs` shows the process's stdout and stderr. `docker exec` starts another process inside a running container. `docker inspect` returns detailed JSON metadata, and `docker stats` reports live CPU and memory usage.

Check state first, then logs, configuration, networking, volumes, and resources. Minimal images may not include `bash`, `curl`, or other debugging tools; use `sh` when available or run a separate diagnostic container. Avoid sharing logs publicly if they contain personal data or secrets.

In [ ]:
debug_commands = [  # Commands for diagnosing a running or stopped container.
    "docker ps -a",  # Start by checking container state.
    "docker logs --tail 100 <container>",  # Read recent logs.
    "docker logs -f <container>",  # Follow new log entries; use Ctrl+C to stop following.
    "docker exec -it <running-container> sh",  # Open a shell in a running container.
    "docker inspect <container>",  # Inspect configuration and state.
    "docker stats",  # View live resource usage.
]  # Finish the diagnostics.
print("\n".join(debug_commands))  # Display diagnostic commands.

## Lesson 7: 7. Port publishing

A service listens on a port inside its container. `-p HOST_PORT:CONTAINER_PORT` publishes that port from the host. In development, binding to `127.0.0.1` can limit access to the local machine; publishing on all host interfaces may make a service accessible to other devices on the network.

The process inside the container usually must listen on `0.0.0.0` for Docker port forwarding to reach it. `EXPOSE` in a Dockerfile documents a port but does not itself publish it.

In [ ]:
port_commands = [  # Demonstrate a localhost-only port mapping.
    "docker run --rm -p 127.0.0.1:8080:80 nginx:alpine",  # Map host loopback 8080 to container port 80.
    "docker ps",  # Confirm the mapping in the PORTS column.
    "docker port <container-name>",  # List a container's port mappings.
]  # Finish commands.
print("\n".join(port_commands))  # Display commands.
print("While the first command runs, open http://localhost:8080 in a browser.")  # Explain how to verify it.

## Lesson 8: 8. Environment variables

Environment variables separate configuration from image contents. They can be supplied with `docker run -e`, an environment file, or Compose's `environment` section. Avoid embedding credentials in a Dockerfile or committing secrets to Git.

An environment file is a convenience for local configuration, not a complete secret-management system. For production, use the platform's secret facility where available and limit who can inspect runtime configuration.

In [ ]:
environment_examples = [  # Examples for local, non-secret configuration.
    "docker run --rm -e APP_ENV=development my-python-app:dev",  # Set one environment variable.
    "docker run --rm --env-file .env.example my-python-app:dev",  # Load a local sample configuration file.
]  # Finish the example commands.
print("\n".join(environment_examples))  # Display commands for review.
example_environment = {"APP_ENV": "development", "REDIS_HOST": "redis", "REDIS_PORT": "6379"}  # Store example environment values.
print(example_environment)  # Display the values that may belong in Compose configuration.
print("Do not put real passwords, API keys, or tokens in the image or public notebook.")  # Add the security reminder.

## Lesson 9: 9. Named volumes

A named volume stores data outside a container's writable layer, letting that data survive container replacement. Databases commonly use volumes for their data directory. Removing a container does not normally remove a named volume.

Persistence is not the same as backup. Volumes can still be deleted or corrupted; important data needs a tested backup and restore process. Volume-removal commands can permanently discard stored data.

In [ ]:
volume_commands = [  # An example lifecycle for disposable volume data.
    "docker volume create demo-data",  # Create a named volume.
    "docker run --rm -v demo-data:/data busybox sh -c 'echo saved > /data/example.txt'",  # Write into the volume from a temporary container.
    "docker run --rm -v demo-data:/data busybox cat /data/example.txt",  # Read the persisted file from another container.
    "docker volume inspect demo-data",  # Inspect volume metadata.
    "docker volume rm demo-data",  # Delete only after confirming this is practice data.
]  # Finish the examples.
print("\n".join(volume_commands))  # Display the workflow.
print("Warning: deleting a volume deletes the data stored in it.")  # Emphasize the destructive consequence.

## Lesson 10: 10. Bind mounts

A bind mount links a specific host path to a container path. It's useful in development because local code changes can be visible inside the container without rebuilding the image. It can also expose host files to the container, so mount only the paths needed.

Named volumes are managed by Docker; bind mounts reference host paths. Prefer the explicit `--mount` syntax when clarity is important. Path syntax differs between PowerShell, Bash, and Linux shells.

In [ ]:
bind_mount_examples = [  # Examples for Linux/macOS Bash; adjust source paths for PowerShell/Windows.
    """docker run --rm --mount type=bind,source="$PWD",target=/workspace,readonly python:3.12-slim python -c 'import os; print(os.listdir("/workspace"))'""",  # Mount the current directory read-only.
    """docker run --rm --mount type=bind,source=/absolute/path/to/project,target=/workspace,readonly python:3.12-slim python -c 'import os; print(os.listdir("/workspace"))'""",  # Use an explicit path.
]  # Finish the examples.
print("\\n".join(bind_mount_examples))  # Display commands for manual adaptation.
print("Use a host path you intend to expose; do not mount sensitive folders unnecessarily.")  # Mention least privilege.

## Lesson 11: 11. Dockerfile instructions

A Dockerfile is a recipe for building an image. Common instructions are `FROM` for a base image, `WORKDIR` for a working directory, `COPY` for copying files, `RUN` for build-time commands, `ENV` for environment defaults, `EXPOSE` for documenting a port, and `CMD` for the default container command.

`RUN` generally executes during image build; `CMD` describes the default process at container start. Prefer exec-form `CMD`, such as `CMD ["python", "app.py"]`, for predictable argument and signal handling.

In [ ]:
dockerfile_text = """FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 5000
CMD ["python", "app.py"]
"""  # Store a minimal Python Dockerfile as text.
print(dockerfile_text)  # Review the instructions before saving them to a Dockerfile.
print("EXPOSE is documentation; use -p or Compose ports to publish a port.")  # Clarify port behavior.

## Lesson 12: 12. Build context and `.dockerignore`

The build context is the set of files available to the builder. In `docker build -t my-app .`, the dot means the current directory is the context. `.dockerignore` excludes files from that context, improving build speed and reducing the chance of copying secrets or local artifacts.

Exclude `.git`, virtual environments, caches, secrets, and unrelated large datasets. Do not ignore files that the Dockerfile needs to copy.

In [ ]:
dockerignore_text = """# Version control and local Python environments
.git
.venv/
venv/
__pycache__/
*.py[cod]

# Local outputs
.ipynb_checkpoints/
.pytest_cache/
dist/
build/

# Secrets and local settings
.env
.env.*
!.env.example
"""  # Define a sample .dockerignore file.
print(dockerignore_text)  # Review the ignore patterns.
print("Build command: docker build -t my-python-app:dev .")  # The final dot sets the build context.

## Lesson 13: 13. Build, tag, and inspect an image

`docker build -t name:tag .` builds and tags an image. After building, inspect the image, check its layers, and run a smoke test. A successful build only establishes that the image build completed; it does not prove the app functions correctly or is secure.

Use descriptive release tags and a controlled update process. A tag like `latest` is mutable and should not be treated as a unique build identity.

In [ ]:
build_workflow = [  # A standard image-build workflow.
    "docker build -t analyst-demo:dev .",  # Build from the current directory.
    "docker image ls analyst-demo",  # Find the image locally.
    "docker image inspect analyst-demo:dev",  # Inspect image metadata.
    "docker image history analyst-demo:dev",  # Review image layers.
    "docker run --rm analyst-demo:dev",  # Run the image's default command if designed for a one-shot run.
]  # Finish workflow.
print("\n".join(build_workflow))  # Display each command for review.

## Lesson 14: 14. Build-cache optimization

Docker can reuse cached build steps if their instructions and relevant inputs have not changed. Copy dependency declarations first, install dependencies, then copy frequently changing application source. This helps avoid reinstalling dependencies whenever a source file changes.

The cache also depends on the build context and builder behavior. Keep `.dockerignore` correct, and inspect the build output to see whether layers were reused.

In [ ]:
cache_order = """# Often cache-friendly:
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .

# Less cache-friendly for frequently changed code:
COPY . .
RUN pip install --no-cache-dir -r requirements.txt
"""  # Compare instruction ordering.
print(cache_order)  # Display the two patterns.
print("The first ordering can reuse the dependency-install layer when only app.py changes.")  # Explain why ordering matters.

## Lesson 15: 15. Containerize a Python web app

To containerize a Python web app, define dependencies, copy them into the image, install them, copy the app, and set a default startup command. The server should normally bind to `0.0.0.0` inside the container. Keep local virtual environments and secrets out of the build context.

The companion project provides a Flask app, dependency file, Dockerfile, `.dockerignore`, and Compose file. Use the README to launch it with Docker Compose.

In [ ]:
from pathlib import Path  # Use Path to inspect companion project files.

project = Path("docker_practice_project")  # The companion project is packaged beside this notebook in the ZIP.
expected = ["app.py", "requirements.txt", "Dockerfile", "compose.yaml", ".dockerignore", "README.md"]  # List expected files.
for filename in expected:  # Inspect each expected project file.
    print(filename, "exists" if (project / filename).exists() else "not found from current directory")  # Report its availability.
print("From the project directory, run: docker compose up --build")  # Display the startup command.

## Lesson 16: 16. ENTRYPOINT versus CMD

`CMD` specifies a default command or arguments. `ENTRYPOINT` specifies the executable the container is designed to run. Used together in exec form, `ENTRYPOINT` commonly names the executable while `CMD` supplies default arguments.

Use `CMD` when users should easily override the default command. Use `ENTRYPOINT` when the image should behave like a particular executable. This design affects how runtime arguments and termination signals are handled.

In [ ]:
patterns = {  # Compare Dockerfile patterns.
    "CMD only": 'CMD ["python", "app.py"]',  # Default command can be overridden at docker run time.
    "ENTRYPOINT + CMD": 'ENTRYPOINT ["python"]\nCMD ["app.py"]',  # Treat the image as a Python command with a default script.
}  # Finish the pattern dictionary.
for name, snippet in patterns.items():  # Display each pattern.
    print(f"{name}:\n{snippet}\n")  # Show the Dockerfile instructions.

## Lesson 17: 17. Docker Compose fundamentals

Compose defines an application stack in YAML, commonly `compose.yaml`. It can declare services, build settings, ports, environment variables, networks, volumes, health checks, and dependencies.

`docker compose up --build` builds and starts services. `docker compose down` removes project containers and networks but preserves named volumes by default. `docker compose down -v` also removes declared volumes and may erase data.

In [ ]:
compose_commands = [  # Commands for a Compose application.
    "docker compose config",  # Render and validate Compose configuration.
    "docker compose up --build",  # Build images and start services.
    "docker compose ps",  # Check service status.
    "docker compose logs -f web",  # Follow web-service logs.
    "docker compose exec web sh",  # Open a shell inside the running web service.
    "docker compose down",  # Stop and remove project containers and network.
]  # Finish the command list.
print("\n".join(compose_commands))  # Display the workflow.
print("Use 'down -v' only when you deliberately want to delete the Compose volumes.")  # Warn about persistence.

## Lesson 18: 18. Networking between services

Compose creates a project network by default. Services can usually reach one another by service name: for example, a web container can connect to Redis at hostname `redis` and port 6379. `localhost` inside the web container means the web container itself, not the Redis container.

Only publish ports that need to be reachable from the host. Services can communicate privately on the Compose network without publishing their ports to the host.

In [ ]:
network_commands = [  # Network inspection and service-discovery commands.
    "docker network ls",  # List networks.
    "docker network inspect <compose-network>",  # Inspect a network and attached containers.
    "docker compose exec web python -c \"import socket; print(socket.gethostbyname('redis'))\"",  # Resolve the Redis service from inside web.
]  # Finish the examples.
print("\n".join(network_commands))  # Display commands for a running Compose project.
print("Use the service name in configuration rather than a container's changing IP address.")  # Explain service discovery.

## Lesson 19: 19. Health checks and dependencies

A health check runs a command to decide whether a container appears healthy. A running process may not yet be ready to serve requests, so health checks can expose startup or dependency problems. Compose can wait for a dependency marked `service_healthy`, but application code should still handle transient failures.

Choose a check that is lightweight and available in the image. A web-process health endpoint may not prove that a downstream database or cache is working.

In [ ]:
healthcheck_yaml = """healthcheck:
  test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://127.0.0.1:5000/health', timeout=2)"]
  interval: 30s
  timeout: 3s
  start_period: 5s
  retries: 3
"""  # Create an example Python-based health check.
dependency_yaml = """depends_on:
  redis:
    condition: service_healthy
"""  # Create an example Compose readiness dependency.
print(healthcheck_yaml)  # Display the health-check block.
print(dependency_yaml)  # Display the depends_on block.

## Lesson 20: 20. PostgreSQL persistence in Compose

Stateful services such as PostgreSQL usually need a named volume mounted to the database data directory. This lets database files persist when the database container is replaced. Environment variables can initialize a local database and user, but production credentials should be supplied securely.

A volume is not a backup. Use a verified backup-and-restore process for important data, and avoid editing live database data files manually.

In [ ]:
postgres_compose = """services:
  db:
    image: postgres:16-alpine
    environment:
      POSTGRES_DB: appdb
      POSTGRES_USER: appuser
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD}
    volumes:
      - postgres_data:/var/lib/postgresql/data

volumes:
  postgres_data:
"""  # Define a Compose service with persistent storage and externally supplied password.
print(postgres_compose)  # Review the YAML.
print("For local learning, use a protected .env file excluded by .dockerignore; never commit real credentials.")  # Explain credential handling.

## Lesson 21: 21. Security: least privilege and secrets

Run as a non-root user when possible, minimize the image, keep dependencies updated, mount only needed host paths, and publish only required ports. Avoid `--privileged` and avoid mounting the Docker socket into an application container: access to that socket can grant control over Docker and potentially the host.

Never hard-code credentials in Dockerfiles or publish them in images. A secret removed in a later image layer may still exist in earlier layers. Use appropriate secret-management facilities and scan dependencies/images as part of release preparation.

In [ ]:
security_review = [  # Use this list during image review.
    "Run the application as a non-root user where possible.",  # Reduce privilege.
    "Keep secrets out of Dockerfiles, image layers, Git, and logs.",  # Protect credentials.
    "Use a maintained base image and update it deliberately.",  # Apply security fixes.
    "Mount only the host directories the app needs.",  # Apply least privilege.
    "Avoid privileged containers and unnecessary Linux capabilities.",  # Reduce access.
    "Publish only required ports.",  # Limit exposure.
    "Scan image dependencies and document any accepted risks.",  # Add a release review step.
]  # Finish the checklist.
for item in security_review:  # Display the checklist.
    print("[ ]", item)  # Start each as unchecked.

## Lesson 22: 22. Non-root user in a Dockerfile

Containers commonly start as root unless the image or Dockerfile changes the user. Add a dedicated user and switch with `USER` before the application starts. Ensure that the user has the permissions it needs to read application files and write only to approved locations.

A non-root process reduces some risks but does not make an image automatically secure. Review file permissions, mounted volumes, kernel capabilities, secrets, and runtime configuration too.

In [ ]:
non_root_snippet = """RUN useradd --create-home appuser && chown -R appuser:appuser /app
USER appuser
CMD ["python", "app.py"]
"""  # Define a common non-root Dockerfile snippet for Debian-based official images.
print(non_root_snippet)  # Display the example.
print("The exact user-creation command depends on the base image; Alpine commonly uses adduser instead.")  # Note portability differences.

## Lesson 23: 23. Multi-stage builds

A multi-stage build uses more than one `FROM` stage. The build stage can compile assets or build packages, while the final stage copies only the runtime artifacts. This can reduce the final image and avoid shipping compilers or build dependencies.

Multi-stage builds are helpful when there are artifacts to copy, but they are not automatically better for every small Python app. Check that all runtime dependencies exist in the final stage, then test and measure the result.

In [ ]:
multi_stage_dockerfile = """FROM python:3.12-slim AS builder
WORKDIR /wheels
COPY requirements.txt .
RUN pip wheel --no-cache-dir --wheel-dir /wheels -r requirements.txt

FROM python:3.12-slim AS runtime
WORKDIR /app
COPY --from=builder /wheels /wheels
COPY requirements.txt .
RUN pip install --no-cache-dir --no-index --find-links=/wheels -r requirements.txt
COPY app.py .
CMD ["python", "app.py"]
"""  # Build wheels in one stage and install them in the runtime stage.
print(multi_stage_dockerfile)  # Display the example.
print("Build and test the final stage specifically; the builder's installed tools are not present automatically.")  # Explain stage isolation.

## Lesson 24: 24. Resource limits and monitoring

Containers share host resources. Options such as `--memory` and `--cpus` can limit resource use for a container. Set limits based on measurements: overly restrictive limits can cause poor performance, timeouts, or out-of-memory termination.

`docker stats` shows live resource use for running containers. Compare observed use to limits and user-facing latency before tuning. Resource configuration syntax may differ between `docker run`, Compose, and orchestration platforms.

In [ ]:
resource_commands = [  # Example commands to constrain and observe a demo container.
    "docker run --rm --memory=256m --cpus=0.50 python:3.12-slim python -c \"print('bounded demo')\"",  # Apply example limits to a short-lived container.
    "docker stats",  # Monitor running containers.
]  # Finish the example list.
print("\n".join(resource_commands))  # Display commands.
print("These values are illustrative; choose limits based on workload measurements.")  # Warn against arbitrary production values.

## Lesson 25: 25. Registries and publishing images

A common release workflow is to build an image, tag it for a registry, authenticate using the registry's approved credential flow, push it, then pull the same tag or digest in the deployment environment. Keep private images and credentials protected.

Avoid putting tokens into shell commands that may be stored in history. Use CI secret stores and least-privilege credentials. Prefer controlled version tags or image digests over depending only on a mutable `latest` tag.

In [ ]:
registry_commands = [  # Template for publishing to a registry.
    "docker login <registry-host>",  # Authenticate through the approved method.
    "docker build -t <registry-host>/<namespace>/<image>:1.0.0 .",  # Build and tag the release.
    "docker push <registry-host>/<namespace>/<image>:1.0.0",  # Upload the image.
    "docker pull <registry-host>/<namespace>/<image>:1.0.0",  # Verify that the image can be downloaded.
    "docker logout <registry-host>",  # End the local login session where applicable.
]  # Finish the publishing workflow.
print("\n".join(registry_commands))  # Show commands with placeholders to replace.
print("Replace placeholders with your registry host, namespace, and project name.")  # Explain customization.

## Lesson 26: 26. Troubleshoot containers systematically

Diagnose systematically rather than changing several settings at once:
1. Is the Docker daemon running?
2. Did the image build successfully?
3. What state is the container in?
4. What do logs show?
5. Are environment variables and mounted paths correct?
6. Are ports published and does the process bind to the right interface?
7. Can the service resolve and reach its dependencies?
8. Are permissions, health checks, or resource limits the cause?

Make one change at a time and record evidence for the root cause.

In [ ]:
diagnostic_steps = [  # A repeatable triage checklist.
    "docker version",  # Check client and daemon communication.
    "docker ps -a",  # Inspect container state.
    "docker logs --tail 100 <container>",  # Read recent output.
    "docker inspect <container>",  # Review environment, mounts, network, and state.
    "docker port <container>",  # Check published ports.
    "docker compose config",  # Validate expanded Compose configuration.
    "docker stats",  # Review resource use.
]  # Finish the checklist.
for command in diagnostic_steps:  # Display the diagnostic order.
    print(command)  # Print one action to check.

## Lesson 27: 27. Clean up safely

Inspect Docker disk use before cleaning up. `docker system df`, container listings, image listings, and volume listings help establish what is using space. Prune commands remove resources within a defined scope and can delete data you intended to keep.

Be especially careful with volumes: they may store database data or application state. Prefer removing named, disposable resources from the practice project rather than applying broad cleanup commands without reviewing their scope.

In [ ]:
cleanup_commands = [  # Inspect before making deletions.
    "docker system df",  # Show Docker disk usage.
    "docker ps -a",  # Inspect containers.
    "docker image ls",  # Inspect images.
    "docker volume ls",  # Inspect persistent volumes.
    "docker container prune",  # Remove all stopped containers after confirmation.
    "docker image prune",  # Remove dangling images after confirmation.
]  # Finish safe-review examples.
print("\n".join(cleanup_commands))  # Display for deliberate manual execution.
print("Do not prune volumes casually; persistent data may be lost.")  # Highlight the biggest risk.

## Lesson 28: 28. Docker in CI/CD

A CI workflow can check out code, run tests, build an image, smoke-test the container, scan dependencies and the image, then publish an approved artifact. Deployment systems can promote the same image through environments, with configuration injected separately.

Keep development bind mounts and debug settings out of production by default. Use a registry, predictable tags, health checks, logs, rollback plans, and secrets provided by the CI/deployment platform.

In [ ]:
ci_steps = [  # Outline a vendor-neutral CI pipeline.
    "Check out version-controlled source code",  # Obtain the exact revision.
    "Run unit tests and quality checks",  # Verify code behavior.
    "Build the Docker image",  # Verify the image build.
    "Start a test container and check its health endpoint",  # Perform a smoke test.
    "Scan dependencies and image layers",  # Inspect for vulnerabilities.
    "Tag and publish the approved image",  # Produce a deployable artifact.
    "Deploy to a staging environment and verify behavior",  # Validate real configuration.
    "Promote carefully and retain a rollback path",  # Reduce deployment risk.
]  # Finish the pipeline outline.
for index, step in enumerate(ci_steps, start=1):  # Number each stage.
    print(f"{index}. {step}")  # Display the pipeline.

## Lesson 29: 29. Docker Compose practice project walkthrough

The companion project has two services: a Flask web app and a Redis service. Compose creates a network, so the app connects to Redis using hostname `redis`. Redis persistence uses the named volume `redis_data`. The web service publishes port 5000 and has a health check.

Run the commands in the companion directory, not from an unrelated directory. The app is for local practice and should not be published directly to the internet without a security and deployment review.

In [ ]:
from pathlib import Path  # Inspect the companion project in the current working directory.
project = Path("docker_practice_project")  # This path exists when the notebook is launched from the extracted pack root.
needed = ["app.py", "requirements.txt", "Dockerfile", "compose.yaml", ".dockerignore", "README.md"]  # List its main files.
for filename in needed:  # Check the list one item at a time.
    print(filename, "exists" if (project / filename).exists() else "not found from this working directory")  # Report project readiness.
print("Run from the project folder: docker compose up --build")  # Show how to start the app.
print("Test /, /health, and /visits at http://localhost:5000.")  # List local endpoints.

## Lesson 30: 30. Final workflow and next steps

A professional Docker workflow combines reproducible builds, useful logs, explicit configuration, persistent storage where required, health checks, minimal privileges, and safe release practices. Docker is one component of application delivery; code testing, monitoring, backups, access controls, and a secure deployment design remain necessary.

To practice, build and run the companion app, inspect its logs, test the health endpoint, visit the Redis-backed counter, stop the services, then start them again and verify that the counter persists in the named volume. Finally, document the architecture and one limitation in the README.

In [ ]:
final_commands = [  # A complete local-practice workflow.
    "docker compose config",  # Validate configuration.
    "docker compose up --build -d",  # Build and start the stack.
    "docker compose ps",  # Check service state.
    "docker compose logs --tail 100 web",  # Read logs.
    "docker compose down",  # Stop containers while preserving named volumes.
    "docker compose up -d",  # Start again and verify persistent state.
]  # Finish the workflow.
print("\n".join(final_commands))  # Display the workflow.
print("Document: what worked, the container networking path, where data persists, and how you verified health.")  # Define a final practice deliverable.